# ЮрТэг — Файнтюнинг Qwen 2.5 7B (Kaggle)

Обучение модели извлечения метаданных из юридических документов.

**Этап 1:** SFT (Supervised Fine-Tuning) на 874 одобренных примерах  
**Этап 2:** DPO (Direct Preference Optimization) на 79 парах правильный/неправильный

---

## Подготовка

1. Создай Kaggle Dataset: New Dataset → загрузи 4 файла из `training/`
2. В настройках ноутбука: Settings → Accelerator → **GPU T4 x2**
3. В правой панели: Add Input → найди свой датасет → Add
4. Запускай ячейки по порядку

In [ ]:
!pip install peft trl bitsandbytes accelerate --quiet

In [ ]:
# Поиск данных
# Kaggle монтирует датасеты в /kaggle/input/<dataset-name>/
# Результаты сохраняются в /kaggle/working/

import os
import glob

matches = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)

if matches:
    DATA_DIR = os.path.dirname(matches[0])
    print(f"Данные найдены: {DATA_DIR}")
    for f in sorted(os.listdir(DATA_DIR)):
        size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024 / 1024
        print(f"  {f} ({size:.1f} МБ)")
else:
    print("ОШИБКА: Данные не найдены!")
    print("1. Создай Kaggle Dataset с файлами: sft_train.jsonl, sft_val.jsonl, dpo_train.jsonl, dpo_val.jsonl")
    print("2. В правой панели ноутбука: Add Input → выбери свой датасет")

## Этап 1: SFT (Supervised Fine-Tuning)

Модель учится извлекать метаданные из юридических документов на 874 одобренных примерах.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/sft_train.jsonl",
    "validation": f"{DATA_DIR}/sft_val.jsonl",
})

print(f"Train: {len(sft_dataset['train'])} примеров")
print(f"Val:   {len(sft_dataset['validation'])} примеров")

example = sft_dataset['train'][0]
print(f"\nПример (роли): {[m['role'] for m in example['messages']]}")
print(f"System: {example['messages'][0]['content'][:100]}...")
print(f"Assistant: {example['messages'][2]['content'][:200]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

sft_train_formatted = sft_dataset["train"].map(apply_chat_template)
sft_val_formatted = sft_dataset["validation"].map(apply_chat_template)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_train_formatted,
    eval_dataset=sft_val_formatted,
    dataset_text_field="text",
    max_seq_length=8192,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="/kaggle/working/sft_checkpoints",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Запускаем SFT обучение...")

In [ ]:
sft_stats = sft_trainer.train()
print(f"\nSFT завершён!")
print(f"Train loss: {sft_stats.training_loss:.4f}")

## Этап 2: DPO (Direct Preference Optimization)

Модель учится на парах: правильный ответ (chosen) vs неправильный (rejected).  
79 пар из реальных ошибок, исправленных человеком.

In [ ]:
import os

dpo_train_path = f"{DATA_DIR}/dpo_train.jsonl"
dpo_val_path = f"{DATA_DIR}/dpo_val.jsonl"

HAS_DPO = os.path.exists(dpo_train_path)
if HAS_DPO:
    print("DPO-данные найдены, запускаем обучение")
else:
    print("DPO-данных нет, пропускаем этап 2 (только SFT)")

In [ ]:
if HAS_DPO:
    from trl import DPOTrainer, DPOConfig

    dpo_dataset = load_dataset("json", data_files={
        "train": dpo_train_path,
        "validation": dpo_val_path,
    })

    print(f"DPO Train: {len(dpo_dataset['train'])} пар")
    print(f"DPO Val:   {len(dpo_dataset['validation'])} пар")

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        tokenizer=tokenizer,
        train_dataset=dpo_dataset["train"],
        eval_dataset=dpo_dataset["validation"],
        args=DPOConfig(
            output_dir="/kaggle/working/dpo_checkpoints",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=5,
            num_train_epochs=2,
            learning_rate=5e-5,
            bf16=True,
            logging_steps=5,
            optim="adamw_8bit",
            seed=42,
            beta=0.1,
            report_to="none",
        ),
    )

    dpo_stats = dpo_trainer.train()
    print(f"\nDPO завершён!")
    print(f"Train loss: {dpo_stats.training_loss:.4f}")
else:
    print("DPO пропущен")

## Быстрый тест

Проверяем модель на коротком тексте перед экспортом.

In [ ]:
model.eval()

test_text = """ДОГОВОР ПОДРЯДА № 15
г. Москва, 10 января 2024 г.

ООО «СтройМонтаж» (Подрядчик) и ИП Фокина Дарья Владимировна (Заказчик)
заключили настоящий договор о нижеследующем:

1. Подрядчик обязуется выполнить ремонтные работы в помещении по адресу:
г. Москва, ул. Ленина, д. 10, офис 5.
2. Стоимость работ: 450 000 руб.
3. Срок выполнения: с 15.01.2024 по 15.03.2024.
4. Неустойка за просрочку: 0.1% от стоимости за каждый день."""

messages = [
    {"role": "system", "content": "Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа. Отвечай СТРОГО чистым JSON."},
    {"role": "user", "content": f"Извлеки метаданные из текста юридического документа.\n\nТекст документа:\n{test_text}"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("Ответ модели:")
print(response)

## Экспорт модели

Сохраняем LoRA-адаптеры и мерженую модель.  
Для GGUF конвертации используем llama.cpp после скачивания.

In [ ]:
# Сохраняем LoRA-адаптеры (маленькие, ~100 МБ)
LORA_PATH = "/kaggle/working/yurteg_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA-адаптеры сохранены в {LORA_PATH}")

In [ ]:
# Мержим LoRA в базовую модель и сохраняем полную версию
from peft import AutoPeftModelForCausalLM

MERGED_PATH = "/kaggle/working/yurteg_merged"

merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)
print(f"Полная модель сохранена в {MERGED_PATH}")

In [ ]:
# Конвертируем в GGUF прямо здесь
!pip install llama-cpp-python --quiet
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama_cpp
!pip install -r /kaggle/working/llama_cpp/requirements/requirements-convert_hf_to_gguf.txt --quiet

!python /kaggle/working/llama_cpp/convert_hf_to_gguf.py \
    /kaggle/working/yurteg_merged \
    --outfile /kaggle/working/yurteg-7b-q4_k_m.gguf \
    --outtype q4_k_m

import os
size = os.path.getsize("/kaggle/working/yurteg-7b-q4_k_m.gguf") / 1024**3
print(f"\nGGUF файл готов: yurteg-7b-q4_k_m.gguf ({size:.1f} ГБ)")
print("Скачай из Output ноутбука после завершения")

## Готово!

### Как скачать

Вкладка **Output** (справа) → скачай `yurteg-7b-q4_k_m.gguf`

### Как запустить на маке

```bash
# 1. Создать Modelfile
cat > Modelfile << 'EOF'
FROM ./yurteg-7b-q4_k_m.gguf

PARAMETER temperature 0
PARAMETER num_ctx 8192

SYSTEM """Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа.
Отвечай СТРОГО чистым JSON."""
EOF

# 2. Создать модель в Ollama
ollama create yurteg -f Modelfile

# 3. Проверить
ollama run yurteg "Извлеки метаданные: Договор аренды между ООО Альфа и ИП Иванов..."
```